# 1. Header, Citations, Purpose & Topic Bridges

**Topic:** Surrogate Models, Bayesian Optimization & Differentiable Proxies
**Category:** 06_smooth_and_physics_informed_ai

### Citations & Core References
- Rasmussen, C. E., & Williams, C. K. I. (2006). *Gaussian Processes for Machine Learning*. MIT Press.
- Jones, D. R., Schonlau, M., & Welch, W. J. (1998). *Efficient Global Optimization of Expensive Black-Box Functions*. J. Global Optimization.
- Srinivas, N., Krause, A., Kakade, S. M., & Seeger, M. (2010). *Gaussian Process Optimization in the Bandit Setting: No Regret and Experimental Design*. ICML.
- Eriksson, D., Pearce, M., Gardner, J., Turner, R. D., & Poloczek, M. (2019). *Scalable Global Optimization via Local Bayesian Optimization (TuRBO)*. NeurIPS.

## 2. Mathematical Foundations & Governing Equations

### Gaussian Process Posterior Equations
Given observations $\mathcal{D} = \{(x_n, y_n)\}_{n=1}^N$:
$$\mu(x_\star) = K(x_\star, X) [K(X, X) + \sigma_n^2 I]^{-1} y$$
$$\sigma^2(x_\star) = k(x_\star, x_\star) - K(x_\star, X) [K(X, X) + \sigma_n^2 I]^{-1} K(X, x_\star)$$

### RBF (Squared Exponential) Covariance Kernel
$$k(x, x') = \sigma_f^2 \exp\left(-\frac{1}{2} \sum_{d=1}^D \frac{(x_d - x'_d)^2}{\ell_d^2}\right)$$

### Acquisition Functions: Expected Improvement (EI) & GP-UCB
$$\alpha_{\text{EI}}(x) = (\mu(x) - y^+ - \xi) \Phi(Z) + \sigma(x) \phi(Z), \quad Z = \frac{\mu(x) - y^+ - \xi}{\sigma(x)}$$
$$\alpha_{\text{UCB}}(x) = \mu(x) + \beta \sigma(x)$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import math
import torch
import numpy as np

torch.manual_seed(42)
N_train, N_test, D = 6, 8, 2

# Non-convex black-box simulation objective: f(x) = sin(x_0) + cos(x_1) - 0.1 * ||x||^2
x_train = torch.randn(N_train, D) * 1.2
y_train = torch.sin(x_train[:, 0]) + torch.cos(x_train[:, 1]) - 0.1 * torch.sum(x_train**2, dim=-1)
x_test = torch.randn(N_test, D) * 1.5

lengthscale = torch.tensor([1.2, 1.2])
variance = torch.tensor(1.5)
noise_var = torch.tensor(1e-3)

print(f"Training points: {N_train}, Query points: {N_test}, Best observed y+: {torch.max(y_train):.4f}")

In [ ]:
# 4. First-Principles Implementation & Native Parity
def rbf_kernel(x1, x2, l, var):
    diff = (x1.unsqueeze(1) - x2.unsqueeze(0)) / l
    dist_sq = torch.sum(diff**2, dim=-1)
    return var * torch.exp(-0.5 * dist_sq)

# 1. Compute Kernel Gram Matrices
K_yy = rbf_kernel(x_train, x_train, lengthscale, variance) + noise_var * torch.eye(N_train)
K_sy = rbf_kernel(x_test, x_train, lengthscale, variance)
K_ss = rbf_kernel(x_test, x_test, lengthscale, variance)

# 2. Gaussian Process Posterior Mean & Variance
alpha = torch.linalg.solve(K_yy, y_train.unsqueeze(-1)).squeeze(-1)
mu_s = K_sy @ alpha
V = torch.linalg.solve(K_yy, K_sy.t())
cov_s = K_ss - K_sy @ V
var_s = torch.clamp(torch.diag(cov_s), min=1e-6)

# 3. Expected Improvement (EI) & Upper Confidence Bound (UCB)
best_y = torch.max(y_train)
sigma_s = torch.sqrt(var_s)
improvement = mu_s - best_y - 0.01
Z = improvement / sigma_s
phi_z = (1.0 / math.sqrt(2.0 * math.pi)) * torch.exp(-0.5 * Z**2)
phi_cdf = 0.5 * (1.0 + torch.erf(Z / math.sqrt(2.0)))
ei_acq = torch.clamp(improvement * phi_cdf + sigma_s * phi_z, min=0.0)
ucb_acq = mu_s + 2.0 * sigma_s

print(f"Posterior mean norm: {torch.norm(mu_s):.4f}, Posterior variance min/max: {torch.min(var_s):.4f}/{torch.max(var_s):.4f}")
print(f"Max Expected Improvement: {torch.max(ei_acq):.4f}, Max UCB: {torch.max(ucb_acq):.4f}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification
# Invariant 1: Variance strictly non-negative
assert torch.all(var_s >= 0.0), "Posterior variance contains negative values"

# Invariant 2: Near-zero residual at training points
mu_train = K_yy @ alpha
train_res = torch.max(torch.abs(mu_train - y_train)).item()
assert train_res < 1e-4, f"Training points not properly fitted: {train_res:.2e}"

# Invariant 3: EI non-negativity
assert torch.all(ei_acq >= 0.0), "Expected improvement contains negative acquisition scores"

print(f"All quantitative invariants verified! Training fit error: {train_res:.2e}")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

iters = 200
t0 = time.perf_counter()
for _ in range(iters):
    _ = rbf_kernel(x_test, x_train, lengthscale, variance)
dt = (time.perf_counter() - t0) / iters * 1000.0
print(f"RBF kernel matrix evaluation latency: {dt:.3f} ms / step")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation
# Pathological: Zero observation noise / jitter on co-located points causes matrix singularity
x_colocated = torch.cat([x_train[:1], x_train[:1]], dim=0)
K_singular = rbf_kernel(x_colocated, x_colocated, lengthscale, variance)
cond_singular = torch.linalg.cond(K_singular).item()
print(f"Pathological co-located condition number: {cond_singular:.2e} (nearly singular)")

# Corrected: Tikhonov jitter (nugget) regularizer guarantees bounded condition number
K_reg = K_singular + 1e-4 * torch.eye(2)
cond_reg = torch.linalg.cond(K_reg).item()
print(f"Corrected regularized condition number: {cond_reg:.4f}")

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Calibrated Epistemic Uncertainty**: Surrogate models guide black-box optimization by quantifying what the model does not know.
2. **Differentiable Acquisition Surfaces**: Analytic kernel gradients allow gradient descent to optimize acquisition functions in high dimensions.
3. **Tikhonov Stability**: Adding a minimal nugget jitter ensures well-conditioned Cholesky factorizations in crowded sample regimes.